# Tutorial: Correlation and Synergy Analysis

In this tutorial, we analyze a cohort of phenopackets associated with the gene **FBN1**. The example data is retrieved from the [phenopacket store](https://github.com/monarch-initiative/phenopacket-store).

Using this dataset, we demonstrate a complete workflow with **ppkt2synergy**, including:

- dataset construction
- correlation analysis
- synergy analysis of phenotypic features

We recommend running this tutorial in a **Jupyter notebook** for interactive exploration and visualization, although it can also be executed as a standard Python script.


## 1. Load phenopackets

We start by loading phenopackets for the **FBN1** cohort:

In [1]:
from ppkt2synergy import load_phenopackets_by_cohort

phenopackets = load_phenopackets_by_cohort("FBN1")

print(f"Loaded {len(phenopackets)} phenopackets")

Loaded 143 phenopackets


The function `load_phenopackets_by_cohort` provides convenient access to publicly available datasets from the phenopacket store.

> **Note：**  
> You can also use your own phenopacket data. As long as your data follows the [GA4GH Phenopacket schema](https://phenopacket-schema.readthedocs.io/en/latest/index.html), it can be directly used in the workflow.

## 2. Build the dataset

Next, we convert phenopackets into a structured dataset suitable for downstream statistical analysis. `build_gpsea_cohort=True` is required if you plan to compute variant-based conditions or perform synergy analysis involving variant effects.

In [ ]:
from ppkt2synergy import PhenotypeDatasetBuilder

dataset = PhenotypeDatasetBuilder(phenopackets).build(
    build_gpsea_cohort = True
)

Individuals Processed: 100%|██████████| 143/143 [00:01<00:00, 83.48 individuals/s]


The parameters specify the transcript of interest, the variant class to include. Detailed explanations are provided in the **Usage** section.

The resulting `dataset` is now ready for correlation and synergy analysis.


## 3. Correlation analysis

Next, we compute pairwise correlations between HPO features across individuals in the cohort.

In [3]:
from ppkt2synergy import HPOCorrelationAnalyzer

analyzer = HPOCorrelationAnalyzer(dataset=dataset)

correlation_results = analyzer.compute_correlation_matrix()

correlation_results.head()

Calculating pairwise correlation: 100%|██████████| 606/606 [00:20<00:00, 29.70it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,p_value_corrected,Count_00,Count_01,Count_10,Count_11,n_individuals,n_pmids,pmids
55,HP:0000218,High palate,HP:0000678,Dental crowding,1.000000,0.000000e+00,0.000000e+00,21,0,0,11,32,1,12203992
397,HP:0004322,Short stature,HP:0004942,Aortic aneurysm,-0.827131,1.078983e-19,2.168755e-17,7,34,33,0,74,5,10756346;11175294;12203992;20375004;21683322
44,HP:0000098,Tall stature,HP:0004942,Aortic aneurysm,0.796637,1.652467e-18,2.214306e-16,45,7,1,26,79,13,10756346;11175294;12203992;20375004;20979188;2...
38,HP:0000098,Tall stature,HP:0002616,Aortic root aneurysm,0.646903,9.895588e-15,9.945066e-13,70,7,10,26,113,18,10756346;11175294;12203992;20375004;20979188;2...
42,HP:0000098,Tall stature,HP:0004322,Short stature,-0.685681,3.243534e-13,2.607801e-11,16,33,37,0,86,5,10756346;11175294;12203992;20375004;21683322


This step identifies pairs of HPO terms that tend to co-occur or show mutually exclusive patterns.

In the result table:

- `HPO_A` and `HPO_B` are the two HPO terms being compared  
- `correlation` indicates the strength and direction of association  
- `p_value_corrected` provides the adjusted significance level  

Interpretation:

- **positive correlation** → the two phenotypes tend to appear together  
- **negative correlation** → the phenotypes tend to occur in different individuals  
- **values near zero** → little or no association  

Detailed descriptions of parameters and output columns are provided in the **Usage** section.

## 4. Visualize correlation results

To better interpret these relationships, we visualize them as a heatmap.

In [4]:
import os
from IPython.display import HTML

STATIC_DIR = "_static" 
os.makedirs(STATIC_DIR, exist_ok=True)

def plotly_html_link(fig, filename, link_text="View interactive heatmap"):
    """
    Generate Plotly HTML file and return an RTD-friendly link.
    """
    filepath = os.path.join(STATIC_DIR, filename)
    fig.write_html(filepath, include_plotlyjs="cdn", full_html=True)
    return HTML(f'<p>To view the interactive plot, click the link below:</p>'
                f'<a href="_static/{filename}" target="_blank">{link_text}</a>')

In [5]:
fig1 = analyzer.plot_correlation_heatmap_with_significance(
    title_name="Cohort FBN1",
)

In [18]:
plotly_html_link(fig1, "correlation_heatmap.html", link_text="Click here to view interactive correlation heatmap")

This visualization highlights the strongest and most statistically significant associations between phenotypic features.

- Strong positive correlations indicate phenotypes that frequently co-occur
- Strong negative correlations indicate phenotypes that tend to occur in different individuals

> **Note：**  
> Hover over the heatmap to see detailed information for each phenotype pair.  
> The thresholds control which interactions are displayed. Lower thresholds include more pairs, while higher thresholds focus on the strongest signals.  

## 5. Inspect available targets

Before running synergy analysis, we inspect the available target variables in the dataset.

In [7]:
diseases_df, sex_df, genes_df, variant_effects_df = dataset.describe_conditions()

diseases_df

,disease_id,label,observed_n,excluded_n
0,OMIM:154700,Marfan syndrome,50,0
1,OMIM:129600,"Ectopia lentis, familial",44,0
2,OMIM:614185,Geleophysic dysplasia 2,19,0
3,OMIM:102370,Acromicric dysplasia,13,0
4,OMIM:616914,Marfan lipodystrophy syndrome,9,0
5,OMIM:184900,Stiff skin syndrome,8,0


In [6]:
sex_df

,sex,n_individuals
0,female,57
1,male,54
2,unknown,32


In [7]:
genes_df

,gene_symbol,n_individuals
0,FBN1,143


In [8]:
variant_effects_df

,MISSENSE_VARIANT,SPLICE_REGION_VARIANT,SPLICE_DONOR_VARIANT,STOP_GAINED,SPLICE_DONOR_5TH_BASE_VARIANT,INTRON_VARIANT,FRAMESHIFT_VARIANT,SPLICE_ACCEPTOR_VARIANT,SYNONYMOUS_VARIANT,SPLICE_DONOR_REGION_VARIANT,INFRAME_INSERTION,INFRAME_DELETION
transcript_id,,,,,,,,,,,,
NM_000138.5,119,6,6,4,1,2,7,2,1,1,1,1
NM_001406716.1,119,6,6,4,1,2,7,2,1,1,1,1
NM_001406717.1,8,1,0,0,0,0,0,0,0,0,0,0


This summary shows which target variables (e.g., disease labels or variant conditions) can be used for downstream analysis.

These targets define the conditions under which phenotype–phenotype relationships are evaluated in the synergy analysis.

In other words, synergy analysis asks whether the association between two phenotypes changes across different conditions (e.g., variant classes or disease groups).


## 6. Synergy analysis

While correlation captures overall associations between phenotypes, it does not account for how these relationships may differ across conditions.

Synergy analysis addresses this by evaluating whether the association between two phenotypes changes depending on a target variable (e.g., variant class or disease group).

We begin by initializing the synergy analyzer:

In [7]:
from ppkt2synergy import SynergyAnalyzer

synergy_analyzer = SynergyAnalyzer(dataset=dataset)

Next, we compute the synergy matrix for the selected target:

In [9]:
from gpsea.model import VariantEffect
condition = dataset.get_variant_effect_condition(
    transcript_id = "NM_000138.5",
    variant_effect=VariantEffect.MISSENSE_VARIANT,
)


In [10]:
synergy_results = synergy_analyzer.compute_synergy_matrix(
    condition=condition
)

synergy_results.head()

Calculating pairwise synergy: 100%|██████████| 298/298 [00:37<00:00,  7.86it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,synergy,p_value,p_value_corrected,Count_00_y0,Count_01_y0,Count_10_y0,Count_11_y0,N_y0,Count_00_y1,Count_01_y1,Count_10_y1,Count_11_y1,N_y1,n_individuals,n_pmids,pmids
34,HP:0000218,High palate,HP:0001519,Disproportionate tall stature,0.091442,0.000999,0.028347,4,3,6,2,15,40,1,6,10,57,72,11,10756346;11175294;12203992;20979188;21594992;2...
144,HP:0001187,Hyperextensibility of the finger joints,HP:0001519,Disproportionate tall stature,0.028053,0.000999,0.028347,4,5,3,0,12,67,4,0,0,71,83,11,10756346;11175294;12203992;20979188;21594992;2...
151,HP:0001382,Joint hypermobility,HP:0001519,Disproportionate tall stature,0.069705,0.000999,0.028347,4,5,5,1,15,67,4,9,10,90,105,11,10756346;11175294;12203992;20979188;21594992;2...
10,HP:0000098,Tall stature,HP:0001187,Hyperextensibility of the finger joints,0.041185,0.000999,0.028347,2,3,6,0,11,58,0,12,0,70,81,12,10756346;11175294;12203992;20979188;21594992;2...
11,HP:0000098,Tall stature,HP:0001382,Joint hypermobility,0.097174,0.000999,0.028347,2,4,6,2,14,58,2,12,17,89,103,12,10756346;11175294;12203992;20979188;21594992;2...


In [13]:
from ppkt2synergy import has_disease
condition_disease = dataset.get_condition(has_disease("OMIM:154700"), name="disease:Marfan syndrome")

In [14]:
synergy_results_disease = synergy_analyzer.compute_synergy_matrix(
    condition=condition_disease
)
synergy_results_disease.head()

Calculating pairwise synergy: 100%|██████████| 298/298 [00:33<00:00,  8.89it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,synergy,p_value,p_value_corrected,Count_00_y0,Count_01_y0,Count_10_y0,Count_11_y0,N_y0,Count_00_y1,Count_01_y1,Count_10_y1,Count_11_y1,N_y1,n_individuals,n_pmids,pmids
186,HP:0002616,Aortic root aneurysm,HP:0003510,Severe short stature,-0.198570,0.000999,0.00383,18,18,0,0,36,12,0,33,0,45,81,5,10756346;11175294;12203992;20375004;21683322
193,HP:0002650,Scoliosis,HP:0030053,Stiff skin,-0.110265,0.000999,0.00383,0,8,0,0,8,27,0,23,0,50,58,4,10756346;11175294;12203992;20375004
116,HP:0001166,Arachnodactyly,HP:0001187,Hyperextensibility of the finger joints,0.118346,0.000999,0.00383,29,0,1,3,33,7,0,19,0,26,59,13,10756346;11175294;12203992;12446365;20979188;2...
148,HP:0001519,Disproportionate tall stature,HP:0001659,Aortic regurgitation,0.084034,0.000999,0.00383,29,2,0,0,31,1,2,9,0,12,43,2,11175294;21683322
189,HP:0002616,Aortic root aneurysm,HP:0008848,Moderately short stature,-0.090761,0.000999,0.00383,27,9,0,0,36,12,0,33,0,45,81,5,10756346;11175294;12203992;20375004;21683322


The resulting table reports pairwise synergy scores between HPO terms with respect to the selected target.

- `HPO_A` and `HPO_B` are the two phenotypes being evaluated
- `synergy` measures how much additional information the pair provides about the target compared to individual features
- `p_value_corrected` indicates statistical significance after multiple testing correction

Interpretation:

- **positive synergy** → the two phenotypes jointly provide additional information about the target  
- **near zero** → the phenotypes contribute largely independently  
- **negative synergy** → the phenotypes are redundant with respect to the target  

Detailed descriptions of all output columns are provided in the **Usage** section.



## 7. Visualize synergy results

We can visualize synergy results as a heatmap.

In [15]:
fig2 = synergy_analyzer.plot_synergy_heatmap(
)

In [12]:
plotly_html_link(fig2, "synergy_heatmap.html", link_text="Click here to view interactive synergy heatmap")

In [16]:
plotly_html_link(fig2, "synergy_heatmap_disease.html", link_text="Click here to view interactive synergy heatmap for Marfan syndrome")

This visualization highlights phenotype pairs that show strong and statistically significant synergy with respect to the selected target.

> **Note：**  
> Hover over the heatmap to see detailed information for each phenotype pair.  
> The thresholds control which interactions are displayed. Lower thresholds include more pairs, while higher thresholds focus on the strongest signals. 

## Summary

In this tutorial, we:

- Loaded phenopacket data  
- Constructed a structured dataset  
- Explored phenotype–phenotype relationships using correlation  
- Identified condition-specific interactions using synergy analysis  

Together, these steps provide a workflow for uncovering both global and condition-dependent relationships between phenotypic features.

For additional usage patterns and parameter options, see the **Usage** section.

In [19]:
import webbrowser
webbrowser.open("file:///D:/storage/Master_Thesis/ppkt2synergy/docs/build/html/index.html")

True